## ECB and CTR modes

### 1. ECB Mode (Electronic Codebook)

* What it stands for: Electronic Codebook
* How it works: It is the most basic, direct way to use a block cipher. The plaintext is split into independent 16-byte blocks, and each block is encrypted separately using the exact same key.
* The Key Characteristic: Same plaintext block = Same ciphertext block. If you have two identical blocks of text in your message, they will produce the exact same encrypted blocks.
* Security Warning: Because of this pattern preservation, ECB is insecure for real-world use (famous for showing the "Tux the Penguin" outline when encrypting images). 

Why use it in class: Because blocks are processed independently and directly, it is the cleanest way to show the avalanche effect (diffusion) on a single block without counter streams getting in the way.


### 2. CTR Mode (Counter Mode)
* What it stands for: Counter Mode
* How it works: It turns AES (a block cipher) into a stream cipher. Instead of encrypting the plaintext directly, the cipher encrypts a changing value (a combination of a unique nonce and an incrementing counter) to generate a pseudo-random keystream. That keystream is then combined with the plaintext using an XOR operation.
* The Key Characteristic: Same plaintext block $\neq$ Same ciphertext block (because the counter changes for every block). It also requires no padding, meaning your ciphertext is the exact same length as your plaintext.
* Security: Highly secure and widely used today (provided you never reuse the same nonce with the same key).

Why use it in class: It is great for teaching modern encryption workflows where you want confidentiality for data streams or arbitrary-length files without worrying about padding.

## Part 1: AES Encryption in Electronic Codebook / ECB Mode

In this first example, we use **AES in Electronic Codebook (ECB) mode** to explore the most basic form of block cipher encryption. ECB is the simplest block cipher mode where plaintext is divided into independent 16-byte blocks and each block is encrypted separately using the same key.

While ECB is generally insecure for real-world applications (because identical plaintext blocks produce identical ciphertext blocks), it is the cleanest way to understand core block-cipher mechanics, padding requirements, and properties like **diffusion** (the avalanche effect).

We will:
1. Import the necessary low-level cipher and padding components.
2. Generate a random AES-128 key.
3. Define a plaintext message.
4. Apply PKCS7 padding to fit the 16-byte block requirement.
5. Encrypt and decrypt the message using AES-ECB.

In [22]:
# Import required libraries
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding  # <-- Added for ECB padding
from cryptography.hazmat.backends import default_backend
import os

### 1. Generate AES-128 Key

AES-128 requires a 128-bit key (16 bytes). We use `os.urandom(16)` to generate a secure random key.

In [23]:
key = os.urandom(16)  # 128-bit key = 16 bytes
print("Key (hex):", key.hex())

Key (hex): 21154adc482d6b70fc27b24ea6b5bddb


### 2. Define the Plaintext

We define the secret message we want to encrypt. Notice that there is no AAD or authentication tag here—just basic encryption.

In [24]:
plaintext = """My name is Edward Joseph Snowden. 
I used to work for the government, but now I work for the public. 
It took me nearly three decades to recognize that there was a 
distinction, and when I did, it got me into a bit of trouble at the office. 
As a result, I now spend my time trying to protect the public from 
the person I used to be—a spy for the Central Intelligence Agency (CIA) 
and National Security Agency (NSA), just another young technologist out 
to build what I was sure would be a better world."""
print("Plaintext:", plaintext)
plaintext = bytes(plaintext, "utf-8")
print(plaintext)

Plaintext: My name is Edward Joseph Snowden. 
I used to work for the government, but now I work for the public. 
It took me nearly three decades to recognize that there was a 
distinction, and when I did, it got me into a bit of trouble at the office. 
As a result, I now spend my time trying to protect the public from 
the person I used to be—a spy for the Central Intelligence Agency (CIA) 
and National Security Agency (NSA), just another young technologist out 
to build what I was sure would be a better world.
b'My name is Edward Joseph Snowden. \nI used to work for the government, but now I work for the public. \nIt took me nearly three decades to recognize that there was a \ndistinction, and when I did, it got me into a bit of trouble at the office. \nAs a result, I now spend my time trying to protect the public from \nthe person I used to be\xe2\x80\x94a spy for the Central Intelligence Agency (CIA) \nand National Security Agency (NSA), just another young technologist out \nto buil

### 3. Encrypt the Plaintext

We initialize the AES cipher in Counter Mode (`modes.CTR`) and use an encryptor object to transform the plaintext into ciphertext.

In [25]:
# 1. Pad the plaintext so its length is a multiple of the 16-byte AES block size
padder = padding.PKCS7(128).padder()
padded_plaintext = padder.update(plaintext) + padder.finalize()

# 2. Initialize the AES cipher in ECB mode (Notice: no nonce is passed here)
cipher = Cipher(algorithms.AES(key), modes.ECB(), backend=default_backend())
encryptor = cipher.encryptor()

# 3. Encrypt the padded plaintext
ciphertext = encryptor.update(padded_plaintext) + encryptor.finalize()
print("Ciphertext (hex):", ciphertext.hex())

Ciphertext (hex): 290ed4a9b6d1ca425415335f750445788d0a8edeac61d25a95c99f597ef1158e5e1219eefbd0ee85483f337ae4853a0f606d8e52a3a9fea0b40c734daa904b8e4ef4bbd53049c4082602ef6b1d68470e4eae775cafd13adf2d912bdcfcbc7a8e0ae97adac06b4791495d6d3ee0963716a5f81eb94fecc92687776c9faf67c5bb3a53e394d6938dfc9f4eb38d01feda0b2b32d65b02a95913b68da1b08fb921dc3c1ec3615ad166b7cf3f47065904c9554fd8f9cf6de69acccea2e9234de9433dced69574f91c82bb0f1595a864c8e26bda52e398d75b86fb1f5d2fb8c7f2de34e991a0e0ce9488ade634f600fa6a4eb107b4328366d9275fb7b8a4b1267c2738d8bde0a7f459e79c5b8d1ca36406a79be518f29facf49700483d915383b00f544a066a42be1481770bb21a0f94cd3213ddb5d84d8f4fd2d1f5d3975951d6138ba5a8973f535b00fed24266500326e7df9f94ec9935e4fa76298ece4b77a1695993118481b5a2b5a9fd7c1fa5ab69ea91301fe70a2d786930bd76ce9a9b8fd5c7526e5231abd47e9c2043247a844235e3b5c25e501a53259cc310b6279740c34bbebaa2dde481eb6c7ece32d154183d40a8b8092953a4fab713c9bd75a54464d62b56aed9f4aa395513caf175c963ea44b08cf9c1edc243e5e28a11531d6542dc29e6cbe8dacdb5ef9af562

### 4. Decrypt the ciphertext

In [26]:
# 1. Create a decryptor object with the same key
decryptor = cipher.decryptor()
decrypted_padded = decryptor.update(ciphertext) + decryptor.finalize()

# 2. Unpad the decrypted data to remove the padding bytes
unpadder = padding.PKCS7(128).unpadder()
decrypted_plaintext = unpadder.update(decrypted_padded) + unpadder.finalize()

print("Decrypted:\n", decrypted_plaintext.decode())

Decrypted:
 My name is Edward Joseph Snowden. 
I used to work for the government, but now I work for the public. 
It took me nearly three decades to recognize that there was a 
distinction, and when I did, it got me into a bit of trouble at the office. 
As a result, I now spend my time trying to protect the public from 
the person I used to be—a spy for the Central Intelligence Agency (CIA) 
and National Security Agency (NSA), just another young technologist out 
to build what I was sure would be a better world.


### 5. Demonstrating Diffusion (The Avalanche Effect)

AES is designed around two core cryptographic properties:
1. **Confusion:** Hiding the relationship between the key and the ciphertext (achieved via SubBytes and permutation).
2. **Diffusion:** Ensuring that a tiny change in the input (like flipping a single character or bit) spreads across the entire output. This is known as the **avalanche effect**. Ideally, changing 1 bit should flip roughly 50% of the output bits.

Let's test this using a single 16-byte block of plaintext with **AES-ECB mode** to see how changing just **one letter** completely transforms the resulting ciphertext.

In [27]:
# 1. Define two nearly identical 16-byte plaintext blocks (differing by only 1 character)
block1 = b"Attack at dawn!!"  # Exactly 16 bytes
block2 = b"Attack at dawn!?"  # Changed '!' to '?' (1 character difference)

# 2. Use our existing AES key
# (Using ECB mode here strictly to show single-block block-cipher diffusion)
cipher = Cipher(algorithms.AES(key), modes.ECB(), backend=default_backend())

# 3. Encrypt Block 1
encryptor1 = cipher.encryptor()
cipher1 = encryptor1.update(block1) + encryptor1.finalize()

# 4. Encrypt Block 2 (1-character change)
encryptor2 = cipher.encryptor()
cipher2 = encryptor2.update(block2) + encryptor2.finalize()

# 5. Compare the results
print("Plaintext 1:  ", block1)
print("Ciphertext 1: ", cipher1.hex())
print("-" * 60)
print("Plaintext 2:  ", block2)
print("Ciphertext 2: ", cipher2.hex())

Plaintext 1:   b'Attack at dawn!!'
Ciphertext 1:  74bcf0bc40d675fe7c656b2e17588320
------------------------------------------------------------
Plaintext 2:   b'Attack at dawn!?'
Ciphertext 2:  52d675c84d4a37bebcca1dc3b9710fbf


### 6. We now show confusion:

In [28]:
# 1. Use the same plaintext block
plaintext_block = b"Attack at dawn!!"  # Exactly 16 bytes

# 2. Define Key A
key_A = b"0123456789abcdef"  # 16 bytes

# 3. Define Key B: Modify just ONE character/byte of Key A ('f' changed to 'e')
key_B = b"0123456789abcdee"  

# 4. Encrypt with Key A
cipher_A = Cipher(algorithms.AES(key_A), modes.ECB(), backend=default_backend())
encryptor_A = cipher_A.encryptor()
ciphertext_A = encryptor_A.update(plaintext_block) + encryptor_A.finalize()

# 5. Encrypt the exact same plaintext with Key B
cipher_B = Cipher(algorithms.AES(key_B), modes.ECB(), backend=default_backend())
encryptor_B = cipher_B.encryptor()
ciphertext_B = encryptor_B.update(plaintext_block) + encryptor_B.finalize()

# 6. Compare the results
print("Plaintext:    ", plaintext_block)
print("-" * 60)
print("Key A:        ", key_A)
print("Ciphertext A: ", ciphertext_A.hex())
print("-" * 60)
print("Key B:        ", key_B, "(Changed last character from 'f' to 'e')")
print("Ciphertext B: ", ciphertext_B.hex())

Plaintext:     b'Attack at dawn!!'
------------------------------------------------------------
Key A:         b'0123456789abcdef'
Ciphertext A:  20a68085dbd272e4549df50079964920
------------------------------------------------------------
Key B:         b'0123456789abcdee' (Changed last character from 'f' to 'e')
Ciphertext B:  2d5a811db79c78189fe09f220dc0e9ad


## Part 2: AES Encryption in Counter Mode

In this example, we use basic **AES in Counter Mode (CTR)** for confidentiality.
CTR mode simply turns the block cipher into a stream cipher. At this stage, we focus solely on encryption, omitting authentication tags, additional data (AAD), or complex modes.

We will:
1. Import the necessary low-level cipher components.
2. Generate a random AES-128 key.
3. Generate a random nonce/counter block.
4. Define a plaintext message.
5. Encrypt and decrypt the message using AES-CTR.

### 2. Generate a Nonce / Counter Block

Counter mode requires a unique nonce (or initialization vector) combined with a counter value to initialize the stream. For AES, this block must match the block size (16 bytes).

In [29]:
nonce = os.urandom(16)  # 16-byte nonce/counter initialization block
print("Nonce (hex):", nonce.hex())

Nonce (hex): 4a9d4de081681a4af2917256b7af6e38


In [30]:
# Initialize the AES cipher in CTR mode
cipher = Cipher(algorithms.AES(key), modes.CTR(nonce), backend=default_backend())
encryptor = cipher.encryptor()

# Encrypt the plaintext
ciphertext = encryptor.update(plaintext) + encryptor.finalize()
print("Ciphertext (hex):", ciphertext.hex())

Ciphertext (hex): d1c51e0c99452595dbc3a4b906eb8fddff35b0e546a877aed0f4f1f25ce263254baf929c2dc95bfd3cf104fd002d522c288f5966a0a53449d9d4191fe8f14bcfb1964389dd63b2e90d675470f86068a9a4356ca59d2ba7203f3e822c36fe45de9272677349a2330d0bdf1ac765f3ca83641ea61707ac8c220fc0f1172ed7295f8c6d6d6c197b1c33de3cc6a0ca1f884923b242b5ff1054487c49b650868a6611f06f6d8bdb57bc30a15d0bc79e1c7b1692057cef56ebcd1b3c85eeea07ab876a770b7db2d803bf9c519371f5bea6b76706926e580b026c9dab25679d6dee5c61acb4d19cf050d7fcf59cc8e6c18c7ab7a1cdb854cd602379c2989ae5c5f10a5766ab90dbd3c8327e22040366b843ce5785ffe53c3cef0179bc2ec3e937b63e72ebda85aeb7eb33104c9f1cade60e4bbd00fffd13439ba141cdae375ae424a3e36ea9841ed9a357134aabb73cee6c9179adb3e7b5e244e80ecd0c5331a7fd4ba0671463b064406213fd9cfc691a8d497e7db46097b9f51c04fc5caf0fc9d3158925ea1dd3e449bfdad731894b180057253d2449ffee5cc0ce7e82a467b42ff2693a261ec693fdb3dd2fc0f34e67a9c0bd0be8b703ad715534a754909e2ff2a6ac72beeb197a336079d710287552275298f718f0f9ded9230f0df5c172bf1069c5862715d9846147019c4a5d

### 5. Decrypt the Ciphertext

To decrypt, we create a decryptor using the same key and nonce. Because CTR mode is symmetric, decryption applies the exact same stream transformation.

In [31]:
# Create a decryptor object with the same key and nonce
decryptor = cipher.decryptor()
decrypted_plaintext = decryptor.update(ciphertext) + decryptor.finalize()

print("Decrypted:", decrypted_plaintext.decode())

Decrypted: My name is Edward Joseph Snowden. 
I used to work for the government, but now I work for the public. 
It took me nearly three decades to recognize that there was a 
distinction, and when I did, it got me into a bit of trouble at the office. 
As a result, I now spend my time trying to protect the public from 
the person I used to be—a spy for the Central Intelligence Agency (CIA) 
and National Security Agency (NSA), just another young technologist out 
to build what I was sure would be a better world.
